In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("affordance_natural.csv")
df = df[df['prompt_type'] == "implicit"]
df.head(3)

,group_id,condition,prompt_type,key_verb,afforded_text,non-afforded_text,related_text,afforded_image,non-afforded_image,related_image
18,1,Brad was sitting in his office when an intrude...,implicit,protect,laptop computer,paper napkin,hunting rifle,laptopcomputer_natural.jpeg,papernapkin_natural.jpeg,huntingrifle_natural.jpeg
19,2,Jenny was hiking in the woods. She left her fr...,implicit,point,pen,nickel,string,pen_natural.jpeg,nickel_natural.jpeg,string_natural.jpeg
20,3,Art was having a party. He realized he didn't ...,implicit,protect,floppy disks,pencils,old magazines,floppydisks_natural.jpeg,pencils_natural.jpeg,oldmagazines_natural.jpeg


In [3]:
df['scenario'] = df['condition']

In [4]:
conditions = ['afforded', 'non-afforded', 'related']

# We'll collect rows in a list of dicts, then convert to DataFrame
output_rows = []

# Enumerate each row in the original CSV
for i, row in df.iterrows():
    # i = row index (0-based)
    # row = the actual row data (scenario, afforded_image, ...)

    scenario_id = row['condition'] 

    # For each group (1..3), we rotate conditions in a Latin Square manner
    for group in range(3):
        # condition index: shift by (i + group)
        condition_idx = (i + group) % 3
        condition_name = conditions[condition_idx]

        # figure out which image column to pull
        # e.g., for 'afforded', we look at row['afforded_image']
        if condition_name == 'afforded':
            image_path = row['afforded_image']
        elif condition_name == 'non-afforded':
            image_path = row['non-afforded_image']
        else:
            image_path = row['related_image']

        image_path = "images/" + image_path

        output_rows.append({
            'list_id'  : group + 1,        # groups labeled 1, 2, 3
            'scenario'  : scenario_id,
            'condition' : condition_name.title(),
            'group_id': row['group_id'],
            'image_path': image_path,
            
        })

# Create a new DataFrame
latin_df = pd.DataFrame(output_rows)

In [5]:
latin_df.groupby("list_id")['condition'].value_counts()

list_id  condition   
1        Afforded        6
         Non-Afforded    6
         Related         6
2        Afforded        6
         Non-Afforded    6
         Related         6
3        Afforded        6
         Non-Afforded    6
         Related         6
Name: count, dtype: int64

In [6]:
latin_df.head(2)

,list_id,scenario,condition,group_id,image_path
0,1,Brad was sitting in his office when an intrude...,Afforded,1,images/laptopcomputer_natural.jpeg
1,2,Brad was sitting in his office when an intrude...,Non-Afforded,1,images/papernapkin_natural.jpeg


In [13]:
latin_df

,list_id,scenario,condition,group_id,image_path
0,1,Brad was sitting in his office when an intrude...,Afforded,1,images/laptopcomputer_natural.jpeg
1,2,Brad was sitting in his office when an intrude...,Non-Afforded,1,images/papernapkin_natural.jpeg
2,3,Brad was sitting in his office when an intrude...,Related,1,images/huntingrifle_natural.jpeg
3,1,Jenny was hiking in the woods. She left her fr...,Non-Afforded,2,images/nickel_natural.jpeg
4,2,Jenny was hiking in the woods. She left her fr...,Related,2,images/string_natural.jpeg
5,3,Jenny was hiking in the woods. She left her fr...,Afforded,2,images/pen_natural.jpeg
6,1,Art was having a party. He realized he didn't ...,Related,3,images/oldmagazines_natural.jpeg
7,2,Art was having a party. He realized he didn't ...,Afforded,3,images/floppydisks_natural.jpeg
8,3,Art was having a party. He realized he didn't ...,Non-Afforded,3,images/pencils_natural.jpeg
9,1,Mike was freezing in the wind. He needed somet...,Afforded,4,images/newspaper_natural.jpeg


In [14]:
latin_df.to_csv("latin_square_natural.csv", index = False, sep='\t')

In [34]:
df_long = pd.melt(df[['scenario', 'group_id', 'afforded_image', 'non-afforded_image', 'related_image']],
        id_vars=["scenario", "group_id"],
                  var_name="Condition", value_name="image_path")
df_long['Condition'] = df_long['Condition'].apply(lambda x: x.split("_")[0].title())
df_long['image_path'] = df_long['image_path'].apply(lambda x: "images/" + x)
df_long.head(3)

,scenario,group_id,Condition,image_path
0,Brad was sitting in his office when an intrude...,1,Afforded,images/laptopcomputer_natural.jpeg
1,Jenny was hiking in the woods. She left her fr...,2,Afforded,images/pen_natural.jpeg
2,Art was having a party. He realized he didn't ...,3,Afforded,images/floppydisks_natural.jpeg


In [35]:
df_merged = pd.merge(latin_df, df_long)

In [36]:
df_merged

,list_id,scenario,condition,group_id,image_path,Condition
0,1,Brad was sitting in his office when an intrude...,Afforded,1,images/laptopcomputer_natural.jpeg,Afforded
1,2,Brad was sitting in his office when an intrude...,Non-Afforded,1,images/papernapkin_natural.jpeg,Non-Afforded
2,3,Brad was sitting in his office when an intrude...,Related,1,images/huntingrifle_natural.jpeg,Related
3,1,Jenny was hiking in the woods. She left her fr...,Non-Afforded,2,images/nickel_natural.jpeg,Non-Afforded
4,2,Jenny was hiking in the woods. She left her fr...,Related,2,images/string_natural.jpeg,Related
5,3,Jenny was hiking in the woods. She left her fr...,Afforded,2,images/pen_natural.jpeg,Afforded
6,1,Art was having a party. He realized he didn't ...,Related,3,images/oldmagazines_natural.jpeg,Related
7,2,Art was having a party. He realized he didn't ...,Afforded,3,images/floppydisks_natural.jpeg,Afforded
8,3,Art was having a party. He realized he didn't ...,Non-Afforded,3,images/pencils_natural.jpeg,Non-Afforded
9,1,Mike was freezing in the wind. He needed somet...,Afforded,4,images/newspaper_natural.jpeg,Afforded
